In [ ]:
# ================== INICIALIZACIÓN COMPLETA DEL ENTORNO ==================
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import networkx as nx
import time
import requests

# Funciones de optimización TSP
def obtener_matriz_distancias_osrm(puntos):
    n = len(puntos)
    matriz = [[0]*n for _ in range(n)]
    url_base = 'http://router.project-osrm.org/route/v1/driving/'
    for i in range(n):
        for j in range(n):
            if i == j:
                matriz[i][j] = 0
            else:
                coords = f"{puntos[i][0]},{puntos[i][1]};{puntos[j][0]},{puntos[j][1]}"
                url = url_base + coords + '?overview=false'
                try:
                    r = requests.get(url)
                    data = r.json()
                    matriz[i][j] = data['routes'][0]['distance']/1000 if 'routes' in data and data['routes'] else float('inf')
                except:
                    matriz[i][j] = float('inf')
                time.sleep(0.1)
    return matriz

def resolver_tsp(matriz):
    n = len(matriz)
    G = nx.Graph()
    for i in range(n):
        for j in range(n):
            if i != j:
                G.add_edge(i, j, weight=matriz[i][j])
    ciclo = nx.approximation.traveling_salesman_problem(G, weight='weight', cycle=True)
    distancia_total = sum(matriz[ciclo[i]][ciclo[i+1]] for i in range(len(ciclo)-1))
    return ciclo, distancia_total

# Carga de datos y widgets
ARCHIVO_DATOS = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Clientes_Ruta.xlsx'
df = pd.read_excel(ARCHIVO_DATOS, sheet_name=None)
df_clientes = df[list(df.keys())[0]]
df_cdr = df[list(df.keys())[1]]

# Mostrar cantidad de clientes por ruta
rutas_disponibles = sorted(df_clientes['Ruta'].dropna().unique())
clientes_por_ruta = {ruta: df_clientes[df_clientes['Ruta'] == ruta].shape[0] for ruta in rutas_disponibles}
rutas_labels = [f"{ruta} ({clientes_por_ruta[ruta]} clientes)" for ruta in rutas_disponibles]

# Widget de selección de rutas
rutas_widget = widgets.SelectMultiple(options=list(zip(rutas_labels, rutas_disponibles)), value=tuple(rutas_disponibles), description='Rutas a optimizar:', style={'description_width': 'initial'}, layout=widgets.Layout(width='60%'))

# Widget de selección de METs con casillas
mets_disponibles = sorted(df_cdr['CodMET'].dropna().unique())
met_checkboxes = [widgets.Checkbox(value=False, description=str(met), indent=False) for met in mets_disponibles]
met_box = widgets.VBox([widgets.Label('Selecciona uno o varios METs:')] + met_checkboxes)

# Widget para definir cantidad de clientes a procesar
clientes_a_procesar = widgets.IntText(value=10, description='Cantidad de clientes a procesar:', style={'description_width': 'initial'}, layout=widgets.Layout(width='30%'))

select_all_btn = widgets.Button(description='Seleccionar todo', button_style='info')
deselect_all_btn = widgets.Button(description='Deseleccionar todo', button_style='warning')
param_button = widgets.Button(description='Aplicar selección', button_style='success')
output_param = widgets.Output()

def seleccionar_todo(b):
    rutas_widget.value = tuple(rutas_disponibles)

def deseleccionar_todo(b):
    rutas_widget.value = tuple()

def aplicar_parametros(b):
    with output_param:
        clear_output()
        mets_seleccionados = [met.description for met in met_checkboxes if met.value]
        print(f'MET(s) seleccionados: {mets_seleccionados}')
        rutas_seleccionadas = list(rutas_widget.value)
        print(f'Rutas a optimizar: {rutas_seleccionadas}')
        print(f'Cantidad de clientes a procesar: {clientes_a_procesar.value}')

select_all_btn.on_click(seleccionar_todo)
deselect_all_btn.on_click(deseleccionar_todo)
param_button.on_click(aplicar_parametros)

# Mostrar el display interactivo
display(widgets.VBox([met_box, rutas_widget, clientes_a_procesar, widgets.HBox([select_all_btn, deselect_all_btn]), param_button, output_param]))
# Los parámetros seleccionados estarán en met_checkboxes, rutas_widget.value y clientes_a_procesar.value para usarse en la optimización.

In [2]:
# ================== EXPORTAR RECORRIDO ÓPTIMO TSP GRUPAL POR MET A EXCEL Y MAPA ==================
import openpyxl
import os
from folium.features import CustomIcon, DivIcon
from datetime import datetime
from branca.element import Template, MacroElement
import folium

# Exporta el recorrido óptimo grupal por MET a un archivo Excel y HTML con detalles, procesando solo la cantidad de clientes indicada.
output_dir = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Outputs'
os.makedirs(output_dir, exist_ok=True)
icon_met_path = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\95_24.png'
mets_seleccionados = [met.description for met in met_checkboxes if met.value]
rutas_seleccionadas = list(rutas_widget.value)
num_clientes = clientes_a_procesar.value
fecha_hora = datetime.now().strftime('%Y%m%d_%H%M%S')

for met in mets_seleccionados:
    # Combinar todos los clientes de las rutas seleccionadas
    clientes_ruta_total = df_clientes[df_clientes['Ruta'].isin(rutas_seleccionadas)]
    if clientes_ruta_total.empty:
        print(f'No hay clientes para las rutas seleccionadas.')
        continue
    if num_clientes > len(clientes_ruta_total):
        print(f'Cantidad solicitada ({num_clientes}) supera los clientes disponibles ({len(clientes_ruta_total)}). Se procesarán todos los disponibles.')
    clientes_ruta = clientes_ruta_total.head(num_clientes)
    met_row = df_cdr[df_cdr['CodMET'] == met]
    if met_row.empty:
        print(f'No se encontró información para el MET {met}.')
        continue
    met_row = met_row.iloc[0]
    puntos = [(met_row['U_longitud'], met_row['U_latitud'])] + [(row['U_longitud'], row['U_latitud']) for _, row in clientes_ruta.iterrows()]
    codigos = [met] + list(clientes_ruta['CodCli'])
    matriz = obtener_matriz_distancias_osrm(puntos)
    ciclo, distancia_total = resolver_tsp(matriz)
    ciclo_filtrado = []
    vistos = set()
    for idx, val in enumerate(ciclo):
        if idx == 0 or idx == len(ciclo)-1 or val not in vistos:
            ciclo_filtrado.append(val)
            vistos.add(val)
    if ciclo_filtrado[-1] != 0:
        ciclo_filtrado.append(0)
    recorrido_codigos = [codigos[i] for i in ciclo_filtrado]

    distancias = []
    for i in range(len(ciclo_filtrado)-1):
        distancias.append(matriz[ciclo_filtrado[i]][ciclo_filtrado[i+1]])

    export_rows = []
    for idx, sec in enumerate(ciclo_filtrado):
        distancia_val = distancias[idx] if idx < len(distancias) else ''
        if sec == 0:
            export_rows.append({
                'Secuencia': idx+1,
                'Tipo': 'MET',
                'Codigo': met,
                'Longitud': met_row['U_longitud'],
                'Latitud': met_row['U_latitud'],
                'Nombre': met_row.get('Nombre', ''),
                'Distancia_al_siguiente_km': distancia_val
            })
        else:
            cli_row = clientes_ruta.iloc[sec-1]
            export_rows.append({
                'Secuencia': idx+1,
                'Tipo': 'Cliente',
                'Codigo': cli_row['CodCli'],
                'Longitud': cli_row['U_longitud'],
                'Latitud': cli_row['U_latitud'],
                'Nombre': cli_row.get('Nombre', ''),
                'Distancia_al_siguiente_km': distancia_val
            })
    df_export = pd.DataFrame(export_rows)
    df_export['Recorrido'] = ' -> '.join([str(x) for x in recorrido_codigos])
    df_export['Distancia_total_km'] = distancia_total

    nombre_archivo = os.path.join(output_dir, f'recorrido_optimo_grupal_{met}_{fecha_hora}.xlsx')
    df_export.to_excel(nombre_archivo, index=False)
    print(f'Recorrido óptimo grupal exportado a: {nombre_archivo}')

    mapa = folium.Map(location=[met_row['U_latitud'], met_row['U_longitud']], zoom_start=10, tiles='OpenStreetMap')

    total_clientes = len(clientes_ruta)
    total_rutas = len(rutas_seleccionadas)
    total_clientes_analizados = len(clientes_ruta_total)

    # Título centrado arriba usando folium.Element y get_root().html.add_child
    titulo_html = '''<div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); z-index: 9998; background-color: white; padding: 7px 20px; border: 2px solid #333; border-radius: 12px; box-shadow: 0 2px 8px rgba(0,0,0,0.3);"><h2 style="margin: 0; color: #333; text-align: center; font-size:14px;"><span style='font-size:14px;'>🗺️ MAPA ANALISIS DISTANCIA OPTIMA GRUPAL POR MET</span></h2><p style="margin: 4px 0 0 0; text-align: center; color: #666; font-size: 9px;">Rutas analizadas: <b>{}</b></p></div>'''.format(total_rutas)
    mapa.get_root().html.add_child(folium.Element(titulo_html))

    # Etiqueta de Resumen estilo imagen ejemplo (personalizada)
    distancia_total = distancia_total if isinstance(distancia_total, (int, float)) else 0
    distancia_promedio = distancia_total / total_clientes if total_clientes > 0 else 0
    distancia_maxima = max(distancias) if distancias else 0
    resumen_html = f'''<div style="position: fixed; top: 130px; right: 35px; width: 260px; background-color: white; border: 2px solid #333; z-index: 9997; font-size: 10.5px; padding: 14px; border-radius: 12px; box-shadow: 0 2px 8px rgba(0,0,0,0.3);"><h3 style="margin-top: 0; color: #333; text-align: left; border-bottom: 2px solid #ddd; padding-bottom: 5px; font-size:13px;"><span style='font-size:13px;'>📊 RESULTADOS DE DISTANCIA</span></h3><p style="margin: 5px 0; font-weight: bold; color: #333; font-size:11px;"><span style='font-size:11px;'>🧮 Total clientes: {total_clientes}</span></p><p style="margin: 6px 0 4px 0; color: #444; font-size:10.5px;">🚗 <b>Distancia total:</b> {distancia_total:.2f} km</p><p style="margin: 6px 0 4px 0; color: #444; font-size:10.5px;">🛣️ <b>Distancia por MET:</b> {distancia_total:.2f} km</p><p style="margin: 6px 0 4px 0; color: #444; font-size:10.5px;">📏 <b>Distancia promedio por cliente:</b> {distancia_promedio:.2f} km</p><p style="margin: 6px 0 4px 0; color: #444; font-size:10.5px;">🏆 <b>Distancia máxima:</b> {distancia_maxima:.2f} km</p></div>'''
    mapa.get_root().html.add_child(folium.Element(resumen_html))

    folium.Marker(location=[met_row['U_latitud'], met_row['U_longitud']],
                  popup=folium.Popup(f'''<div style="background:#2A81CB; color:#fff; border-radius:14px; box-shadow:0 2px 8px rgba(0,0,0,0.15); padding:14px; border:2px solid #fff; min-width:220px; text-align:center;">
                    <div style="font-size:20px; font-weight:bold; margin-bottom:6px;">🏠 MET: <span style="color:#FFD600;">{met}</span></div>
                    <div style="font-size:15px; color:#fff; margin-bottom:2px;">Coordenadas:<br><span style="font-size:14px; color:#FFD600;">{met_row['U_latitud']}, {met_row['U_longitud']}</span></div>
                  </div>''', max_width=260),
                  icon=CustomIcon(icon_met_path, icon_size=(32,32))).add_to(mapa)

    # Clientes con número de secuencia y etiquetas bonitas (secuencia real del recorrido)
    secuencia_cliente = 1
    for idx, sec in enumerate(ciclo_filtrado[1:]):
        if sec == 0:
            continue  # Saltar MET
        cli_row = clientes_ruta.iloc[sec-1]
        codcli = cli_row['CodCli']
        nombre = cli_row.get('Nombre', '')
        ruta_cliente = cli_row.get('Ruta', '')
        # Distancia anterior: del punto anterior (incluyendo MET) al actual
        idx_ciclo = idx + 1  # ciclo_filtrado[1:] corresponde a idx+1 en ciclo_filtrado
        distancia_anterior = matriz[ciclo_filtrado[idx_ciclo-1]][ciclo_filtrado[idx_ciclo]] if idx_ciclo > 0 else 'N/A'
        # Distancia siguiente: del actual al siguiente punto
        distancia_siguiente = matriz[ciclo_filtrado[idx_ciclo]][ciclo_filtrado[idx_ciclo+1]] if idx_ciclo+1 < len(ciclo_filtrado) else 'N/A'

        popup_html = f'''
        <div style=\"background: #fff; border-radius: 16px; box-shadow: 0 2px 8px rgba(0,0,0,0.15); padding: 14px; border: 2px solid #8BC34A; min-width: 260px; max-width: 340px;\">
            <div style=\"font-size:20px; color:#7CB342; font-weight:bold; margin-bottom:4px;\">
                <span style=\"vertical-align:middle;\">👨‍💼</span> Cliente: <span style=\"color:#7CB342;\">{codcli}</span>
            </div>
            <div style=\"font-size:16px; color:#333; margin-bottom:4px;\">
                <span style=\"vertical-align:middle;\">📚</span> <b>Nombre:</b> {nombre}
            </div>
            <div style=\"font-size:15px; color:#2A81CB; margin-bottom:4px;\">
                <span style=\"vertical-align:middle;\">🗺️</span> <b>Ruta:</b> {ruta_cliente}
            </div>
            <div style=\"font-size:15px; color:#CB2A2A; margin-bottom:4px;\">
                <span style=\"vertical-align:middle;\">↩️</span> <b>Distancia anterior:</b> {distancia_anterior:.2f} km
            </div>
            <div style=\"font-size:15px; color:#2A81CB; margin-bottom:4px;\">
                <span style=\"vertical-align:middle;\">➡️</span> <b>Distancia siguiente:</b> {distancia_siguiente:.2f} km
            </div>
            <div style=\"font-size:15px; color:#FFC107; margin-bottom:4px;\">
                <span style=\"vertical-align:middle;\">🔢</span> <b>Número de secuencia:</b> {secuencia_cliente}
            </div>
        </div>
        '''
        folium.Marker(
            location=[cli_row['U_latitud'], cli_row['U_longitud']],
            popup=folium.Popup(popup_html, max_width=340),
            icon=folium.Icon(color='blue', icon='info-sign')
        ).add_to(mapa)
        folium.Marker(
            location=[cli_row['U_latitud'], cli_row['U_longitud']],
            icon=DivIcon(
                icon_size=(24,24),
                icon_anchor=(12,12),
                html=f'<div style="font-size:16px; color:white; background:#CB2A2A; border-radius:50%; width:24px; height:24px; text-align:center; line-height:24px; border:2px solid #fff;">{secuencia_cliente}</div>'
            ),
        ).add_to(mapa)
        secuencia_cliente += 1

    ruta_coords = [puntos[i][::-1] for i in ciclo_filtrado]
    folium.PolyLine(ruta_coords, color='green', weight=5, opacity=0.8, tooltip=f"Recorrido óptimo grupal MET {met}").add_to(mapa)
    nombre_mapa = os.path.join(output_dir, f'mapa_recorrido_optimo_grupal_{met}_{fecha_hora}.html')
    mapa.save(nombre_mapa)
    print(f'Mapa grupal exportado a: {nombre_mapa}')

Recorrido óptimo grupal exportado a: C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Outputs\recorrido_optimo_grupal_MET Celaya_20250829_164907.xlsx
Mapa grupal exportado a: C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Outputs\mapa_recorrido_optimo_grupal_MET Celaya_20250829_164907.html
Recorrido óptimo grupal exportado a: C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Outputs\recorrido_optimo_grupal_MET Querétaro_20250829_164907.xlsx
Mapa grupal exportado a: C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Outputs\mapa_recorrido_optimo_grupal_MET Querétaro_20250829_164907.html
